In [2]:
from pyspark.sql import functions as F

# ============================================
# ÉTAPE 0 : LECTURE DES DONNÉES
# ============================================
storage_account = "energybigdatastorage"
container_raw = "raw"

path_raw = f"abfss://{container_raw}@{storage_account}.dfs.core.windows.net/energy_data_extracted/archive (3).zip/acorn_details.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("encoding", "ISO-8859-1") \
    .load(path_raw)

print(f"Nombre de lignes : {df.count()}")
print(f"Nombre de colonnes : {len(df.columns)}")
df.printSchema()
df.show(5)

In [3]:
# ============================================
# ÉTAPE 1 : SUPPRIMER LES DOUBLONS
# ============================================
nb_avant = df.count()
df = df.dropDuplicates()
nb_apres = df.count()
print(f"Lignes avant : {nb_avant}")
print(f"Lignes après : {nb_apres}")
print(f"Doublons supprimés : {nb_avant - nb_apres}")

In [4]:
# ============================================
# ÉTAPE 2 : VÉRIFICATION DES VALEURS NULLES
# ============================================
print("=== VALEURS NULLES PAR COLONNE ===")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [5]:
# Vérifier aussi les valeurs vides en string
print("=== VALEURS VIDES EN STRING ===")
df.select([F.count(F.when(F.col(c) == "", c)).alias(c) for c in df.columns]).show()

# Vérifier spécifiquement REFERENCE
print("=== APERÇU REFERENCE ===")
df.select("REFERENCE").where(F.col("REFERENCE").isNull() | (F.col("REFERENCE") == "")).show()

In [7]:
# Renommer les colonnes avec des espaces
df = df.withColumnRenamed("MAIN CATEGORIES", "MAIN_CATEGORIES")

# Vérifier
df.printSchema()

# Sauvegarder
path_processed = f"abfss://processed@{storage_account}.dfs.core.windows.net/acorn_details/"
df.write.format("delta").mode("overwrite").save(path_processed)
print(" acorn_details sauvegardé dans processed/acorn_details/")